In [ ]:
# Let's now perform the complete retrieval and structuring of USGS gauges for all 50 states.
import pandas as pd
import requests
from io import StringIO

# List of all 50 state codes
states = [
    'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA',
    'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD',
    'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ',
    'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC',
    'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY'
]

# Function to fetch NWIS gauge data for a state
def fetch_usgs_gauges(state_code):
    try:
        url = f"https://waterservices.usgs.gov/nwis/site/?stateCd={state_code}&siteType=ST&format=rdb"
        response = requests.get(url)
        response.raise_for_status()
        data_text = response.text
        data_lines = [line for line in data_text.splitlines() if not line.startswith('#')]
        columns = data_lines[0].split('\t')
        data_str = '\n'.join(data_lines[1:])
        df = pd.read_csv(StringIO(data_str), sep='\t', names=columns, low_memory=False)
        df.dropna(how='all', inplace=True)
        df.reset_index(drop=True, inplace=True)
        df['State'] = state_code
        return df[['agency_cd', 'site_no', 'station_nm', 'dec_lat_va', 'dec_long_va', 'alt_va', 'State']]
    except Exception as e:
        print(f"Error fetching data for state {state_code}: {e}")
        return pd.DataFrame(columns=['agency_cd', 'site_no', 'station_nm', 'dec_lat_va', 'dec_long_va', 'alt_va', 'State'])

# Fetch data for all states
usgs_sites_all_states = pd.concat([fetch_usgs_gauges(state) for state in states], ignore_index=True)

# Rename columns as requested
usgs_sites_all_states.rename(columns={
    'site_no': 'site_id',
    'station_nm': 'name',
    'dec_lat_va': 'latitude',
    'dec_long_va': 'longitude',
    'alt_va': 'elevation'
}, inplace=True)

# Export the combined DataFrame to Excel
usgs_sites_all_states.to_excel('usgs_sites_all_states.xlsx', index=False)


usgs_sites_all_states.head()